# 5.2 Inner Product Spaces

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 5:** Inner Product Spaces

---

### What you will learn

- How to **generalize** the dot product to abstract inner product spaces via four axioms.
- How to define and compute a **weighted dot product** and verify the axioms.
- How to define an **integral inner product** for function spaces and compute it from scratch.
- How **norm** and **distance** extend naturally to any inner product space.
- How **orthogonality** and the **orthogonal complement** generalize beyond $\mathbb{R}^n$.

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.inner import dot, norm, are_orthogonal
from linalg_core import EPSILON

import math

---

## 2. Motivation

The standard dot product $\langle \mathbf{u}, \mathbf{v} \rangle = u_1 v_1 + u_2 v_2 + \cdots + u_n v_n$
works perfectly in $\mathbb{R}^n$. But what happens when our "vectors" are
**polynomials**, **continuous functions**, or **matrices**?

Consider two polynomials $p(t) = 1$ and $q(t) = t$. There is no obvious way
to "multiply component by component" because a polynomial is not a finite
tuple of numbers. Yet in applications (Fourier analysis, approximation theory,
quantum mechanics) we desperately need a notion of **angle**, **length**, and
**orthogonality** in these richer spaces.

The solution: identify the **essential properties** of the dot product
--- the axioms that make everything work --- and declare that *any*
operation satisfying those axioms is an **inner product**. The resulting
structure is called an **inner product space**.

---

## 3. Build --- Core Concepts

### 3.1 The Four Axioms of an Inner Product

Let $V$ be a real vector space. A function
$\langle \cdot, \cdot \rangle : V \times V \to \mathbb{R}$ is an
**inner product** if, for all $\mathbf{u}, \mathbf{v}, \mathbf{w} \in V$
and all scalars $c \in \mathbb{R}$, the following hold:

| # | Axiom | Statement |
|---|---|---|
| 1 | **Symmetry** | $\langle \mathbf{u}, \mathbf{v} \rangle = \langle \mathbf{v}, \mathbf{u} \rangle$ |
| 2 | **Additivity** | $\langle \mathbf{u} + \mathbf{v}, \mathbf{w} \rangle = \langle \mathbf{u}, \mathbf{w} \rangle + \langle \mathbf{v}, \mathbf{w} \rangle$ |
| 3 | **Homogeneity** | $\langle c\,\mathbf{u}, \mathbf{v} \rangle = c\,\langle \mathbf{u}, \mathbf{v} \rangle$ |
| 4 | **Positive Definiteness** | $\langle \mathbf{v}, \mathbf{v} \rangle \geq 0$, with equality if and only if $\mathbf{v} = \mathbf{0}$ |

> **Remark.** Axioms 2 and 3 together say the inner product is **linear in
> the first argument**. Combined with axiom 1 (symmetry) this gives
> linearity in the second argument as well --- the inner product is
> **bilinear**.

The standard dot product on $\mathbb{R}^n$ satisfies all four axioms, so
$(\mathbb{R}^n, \text{dot})$ is an inner product space. The power of the
definition is that *many other* operations also satisfy these axioms.

### 3.2 Weighted Dot Product

A simple generalization of the standard dot product assigns a **positive
weight** to each component:

$$\langle \mathbf{u}, \mathbf{v} \rangle_w
  = w_1 u_1 v_1 + w_2 u_2 v_2 + \cdots + w_n u_n v_n,
  \qquad w_i > 0.$$

This is an inner product on $\mathbb{R}^n$ for any choice of strictly
positive weights. Let us **implement it from scratch** and then
**verify all four axioms** numerically.

In [ ]:
def weighted_dot(u, v, weights):
    """Weighted inner product: sum of w_i * u_i * v_i.

    Parameters
    ----------
    u, v : list[float]
        Vectors of equal length.
    weights : list[float]
        Strictly positive weights, same length as u and v.

    Returns
    -------
    float
    """
    if len(u) != len(v) or len(u) != len(weights):
        raise ValueError("Dimension mismatch")
    if any(w <= 0 for w in weights):
        raise ValueError("All weights must be strictly positive")
    return sum(w * a * b for w, a, b in zip(weights, u, v))


w = [2, 1, 3]
u = [1, 0, -2]
v = [3, 4, 1]

print("Weighted dot <u, v>_w =", weighted_dot(u, v, w))
print("By hand: 2*1*3 + 1*0*4 + 3*(-2)*1 =", 2*1*3 + 1*0*4 + 3*(-2)*1)

In [ ]:
u = [1, 0, -2]
v = [3, 4, 1]
w_vec = [2, 1, 3]
c = 5.0
x = [2, -1, 3]

print("=== Verifying the four axioms for weighted dot ===")
print()

sym_lhs = weighted_dot(u, v, w_vec)
sym_rhs = weighted_dot(v, u, w_vec)
print("Axiom 1 (Symmetry):")
print(f"  <u,v>_w = {sym_lhs}")
print(f"  <v,u>_w = {sym_rhs}")
print(f"  Equal? {abs(sym_lhs - sym_rhs) < EPSILON}")
print()

u_plus_v = [a + b for a, b in zip(u, v)]
add_lhs = weighted_dot(u_plus_v, x, w_vec)
add_rhs = weighted_dot(u, x, w_vec) + weighted_dot(v, x, w_vec)
print("Axiom 2 (Additivity):")
print(f"  <u+v, x>_w = {add_lhs}")
print(f"  <u,x>_w + <v,x>_w = {add_rhs}")
print(f"  Equal? {abs(add_lhs - add_rhs) < EPSILON}")
print()

cu = [c * a for a in u]
hom_lhs = weighted_dot(cu, v, w_vec)
hom_rhs = c * weighted_dot(u, v, w_vec)
print("Axiom 3 (Homogeneity):")
print(f"  <cu, v>_w = {hom_lhs}")
print(f"  c * <u,v>_w = {hom_rhs}")
print(f"  Equal? {abs(hom_lhs - hom_rhs) < EPSILON}")
print()

pos_val = weighted_dot(u, u, w_vec)
zero = [0, 0, 0]
pos_zero = weighted_dot(zero, zero, w_vec)
print("Axiom 4 (Positive Definiteness):")
print(f"  <u,u>_w = {pos_val}  (>= 0? {pos_val >= 0})")
print(f"  <0,0>_w = {pos_zero}  (== 0? {abs(pos_zero) < EPSILON})")
print(f"  <u,u>_w > 0 for u != 0? {pos_val > 0}")

### 3.3 Integral Inner Product

Let $C[a, b]$ denote the vector space of continuous real-valued functions
on $[a, b]$. The **integral inner product** is

$$\langle f, g \rangle = \int_a^b f(t)\, g(t)\, dt.$$

This satisfies the four axioms (the proof uses properties of the Riemann
integral). In particular:

- **Symmetry** follows from $f(t) g(t) = g(t) f(t)$.
- **Positive definiteness** uses the fact that $f(t)^2 \geq 0$ and
  $\int_a^b f(t)^2\, dt = 0$ implies $f \equiv 0$ for continuous $f$.

We implement this from scratch using the **trapezoidal rule** for
numerical integration:

$$\int_a^b h(t)\, dt \approx \frac{\Delta t}{2}
  \left[ h(t_0) + 2 h(t_1) + 2 h(t_2) + \cdots + 2 h(t_{N-1}) + h(t_N) \right],
  \qquad \Delta t = \frac{b - a}{N}.$$

In [ ]:
def trapezoidal(f, a, b, n=1000):
    """Approximate the integral of f from a to b using the trapezoidal rule.

    Parameters
    ----------
    f : callable
        Function of one variable.
    a, b : float
        Integration bounds.
    n : int
        Number of sub-intervals (default 1000).

    Returns
    -------
    float
    """
    dt = (b - a) / n
    total = 0.5 * f(a) + 0.5 * f(b)
    for i in range(1, n):
        total += f(a + i * dt)
    return total * dt


def integral_ip(f, g, a=0.0, b=1.0, n=1000):
    """Integral inner product <f, g> = integral from a to b of f(t)*g(t) dt.

    Parameters
    ----------
    f, g : callable
        Functions of one variable.
    a, b : float
        Integration interval (default [0, 1]).
    n : int
        Number of sub-intervals for trapezoidal rule.

    Returns
    -------
    float
    """
    return trapezoidal(lambda t: f(t) * g(t), a, b, n)


p = lambda t: 1.0
q = lambda t: t
r = lambda t: t ** 2

print("Integral inner products on [0, 1]:")
print(f"  <1, t>   = {integral_ip(p, q):.6f}   (exact: 1/2 = 0.5)")
print(f"  <t, t>   = {integral_ip(q, q):.6f}   (exact: 1/3 = {1/3:.6f})")
print(f"  <1, t^2> = {integral_ip(p, r):.6f}   (exact: 1/3 = {1/3:.6f})")
print(f"  <t, t^2> = {integral_ip(q, r):.6f}   (exact: 1/4 = 0.25)")

### 3.4 Norm and Distance in General Inner Product Spaces

Given any inner product $\langle \cdot, \cdot \rangle$, we define the
**norm** (generalized length) and **distance** exactly as in $\mathbb{R}^n$:

$$\| \mathbf{v} \| = \sqrt{\langle \mathbf{v}, \mathbf{v} \rangle},
  \qquad d(\mathbf{u}, \mathbf{v}) = \| \mathbf{u} - \mathbf{v} \|.$$

These definitions automatically inherit all the familiar properties
(non-negativity, $\|c \mathbf{v}\| = |c|\,\|\mathbf{v}\|$, triangle
inequality) from the inner product axioms.

Let us compute norms and distances under the **weighted** inner product
and the **integral** inner product.

In [ ]:
def weighted_norm(v, weights):
    """Norm induced by weighted inner product."""
    return math.sqrt(weighted_dot(v, v, weights))


def weighted_distance(u, v, weights):
    """Distance induced by weighted inner product."""
    diff = [a - b for a, b in zip(u, v)]
    return weighted_norm(diff, weights)


w = [2, 1, 3]
u = [1, 0, -2]
v = [3, 4, 1]

print("Weighted norm and distance (weights = [2, 1, 3]):")
print(f"  ||u||_w = sqrt(<u,u>_w) = sqrt({weighted_dot(u, u, w)}) = {weighted_norm(u, w):.6f}")
print(f"  ||v||_w = sqrt(<v,v>_w) = sqrt({weighted_dot(v, v, w)}) = {weighted_norm(v, w):.6f}")
print(f"  d_w(u, v) = {weighted_distance(u, v, w):.6f}")
print()

print("Compare with standard norm (all weights = 1):")
print(f"  ||u|| = {norm(u):.6f}")
print(f"  ||v|| = {norm(v):.6f}")
print()


def integral_norm(f, a=0.0, b=1.0, n=1000):
    """Norm induced by integral inner product."""
    return math.sqrt(integral_ip(f, f, a, b, n))


def integral_distance(f, g, a=0.0, b=1.0, n=1000):
    """Distance induced by integral inner product."""
    diff = lambda t: f(t) - g(t)
    return integral_norm(diff, a, b, n)


p = lambda t: 1.0
q = lambda t: t
r = lambda t: t ** 2

print("Integral norms on [0, 1]:")
print(f"  ||1||   = {integral_norm(p):.6f}   (exact: 1.0)")
print(f"  ||t||   = {integral_norm(q):.6f}   (exact: 1/sqrt(3) = {1/math.sqrt(3):.6f})")
print(f"  ||t^2|| = {integral_norm(r):.6f}   (exact: 1/sqrt(5) = {1/math.sqrt(5):.6f})")
print()
print(f"  d(1, t)   = {integral_distance(p, q):.6f}")
print(f"  d(t, t^2) = {integral_distance(q, r):.6f}")

### 3.5 Orthogonality in Inner Product Spaces

Two vectors $\mathbf{u}$ and $\mathbf{v}$ are **orthogonal** with respect
to an inner product if

$$\langle \mathbf{u}, \mathbf{v} \rangle = 0.$$

Orthogonality depends on the choice of inner product! Two vectors can be
orthogonal under one inner product and not under another.

**Example:** Find two polynomials that are orthogonal under the integral
inner product on $[0, 1]$.

Consider $f(t) = 1$ and $g(t) = t - \frac{1}{2}$. Then:

$$\langle f, g \rangle = \int_0^1 1 \cdot \left(t - \tfrac{1}{2}\right) dt
  = \left[ \frac{t^2}{2} - \frac{t}{2} \right]_0^1
  = \left(\frac{1}{2} - \frac{1}{2}\right) - 0 = 0.$$

In [ ]:
f = lambda t: 1.0
g = lambda t: t - 0.5

ip_fg = integral_ip(f, g, 0, 1)
print("Orthogonality under integral IP on [0, 1]:")
print(f"  f(t) = 1")
print(f"  g(t) = t - 1/2")
print(f"  <f, g> = {ip_fg:.10f}")
print(f"  Orthogonal? {abs(ip_fg) < 1e-6}")
print()

h = lambda t: t ** 2 - t + 1.0 / 6.0
ip_fh = integral_ip(f, h, 0, 1)
ip_gh = integral_ip(g, h, 0, 1)
print("A third polynomial orthogonal to both f and g:")
print(f"  h(t) = t^2 - t + 1/6")
print(f"  <f, h> = {ip_fh:.10f}  (orthogonal to f? {abs(ip_fh) < 1e-6})")
print(f"  <g, h> = {ip_gh:.10f}  (orthogonal to g? {abs(ip_gh) < 1e-6})")
print()

print("Contrast: standard dot product cannot see this relationship.")
print("Orthogonality is a property of the inner product, not the vectors alone.")

### 3.6 Orthogonal Complement

Let $W$ be a subspace of an inner product space $V$. The **orthogonal
complement** of $W$ is

$$W^\perp = \{ \mathbf{v} \in V \mid \langle \mathbf{v}, \mathbf{w} \rangle = 0
  \text{ for all } \mathbf{w} \in W \}.$$

Key properties:

1. $W^\perp$ is itself a subspace of $V$.
2. $V = W \oplus W^\perp$ &emsp; (every vector decomposes uniquely).
3. $\dim(W) + \dim(W^\perp) = \dim(V)$.
4. $(W^\perp)^\perp = W$.

**Example in $\mathbb{R}^4$:** Let $W = \text{span}\{\mathbf{w}_1, \mathbf{w}_2\}$
with the standard dot product. To find $W^\perp$, we need all vectors
$\mathbf{v} = (v_1, v_2, v_3, v_4)$ satisfying
$\langle \mathbf{v}, \mathbf{w}_i \rangle = 0$ for each basis vector of $W$.
This gives a homogeneous system whose null space is $W^\perp$.

In [ ]:
def orthogonal_complement_basis(W_basis):
    """Find a basis for W^perp in R^n using the standard dot product.

    Parameters
    ----------
    W_basis : list[list[float]]
        Basis vectors of W as rows.

    Returns
    -------
    list[list[float]]
        Basis vectors of W^perp.

    Method: Row-reduce the matrix whose rows are the W_basis vectors,
    then read off the null space from the RREF.
    """
    m = len(W_basis)
    if m == 0:
        return []
    n = len(W_basis[0])

    A = [row[:] for row in W_basis]

    pivot_cols = []
    current_row = 0
    for col in range(n):
        pivot = None
        for row in range(current_row, m):
            if abs(A[row][col]) > EPSILON:
                pivot = row
                break
        if pivot is None:
            continue

        A[current_row], A[pivot] = A[pivot], A[current_row]

        scale = A[current_row][col]
        A[current_row] = [x / scale for x in A[current_row]]

        for row in range(m):
            if row != current_row and abs(A[row][col]) > EPSILON:
                factor = A[row][col]
                A[row] = [A[row][j] - factor * A[current_row][j] for j in range(n)]

        pivot_cols.append(col)
        current_row += 1

    free_cols = [j for j in range(n) if j not in pivot_cols]

    basis = []
    for fc in free_cols:
        vec = [0.0] * n
        vec[fc] = 1.0
        for i, pc in enumerate(pivot_cols):
            vec[pc] = -A[i][fc]
        basis.append(vec)

    return basis


w1 = [1, 2, 0, 1]
w2 = [0, 1, 1, 0]
W_basis = [w1, w2]

W_perp = orthogonal_complement_basis(W_basis)

print(f"W = span{{ {w1}, {w2} }}")
print(f"dim(W) = {len(W_basis)}")
print(f"dim(W^perp) = {len(W_perp)}")
print(f"dim(W) + dim(W^perp) = {len(W_basis) + len(W_perp)}  (should equal n = 4)")
print()
print("Basis for W^perp:")
for i, v in enumerate(W_perp):
    print(f"  v{i+1} = {v}")

print()
print("Verification --- every W^perp basis vector is orthogonal to every W basis vector:")
for i, v in enumerate(W_perp):
    for j, w in enumerate(W_basis):
        d = dot(v, w)
        print(f"  <v{i+1}, w{j+1}> = {d:.6f}  (orthogonal? {abs(d) < EPSILON})")

---

## 4. Exercises

Test your understanding with the following problems. Write your solutions
in the code cells provided.

### Exercise 1 (Easy)

Let $\mathbf{u} = (2, -1, 3)$, $\mathbf{v} = (1, 4, -2)$, and weights
$\mathbf{w} = (1, 2, 5)$.

1. Compute $\langle \mathbf{u}, \mathbf{v} \rangle_w$ by hand, then verify
   with `weighted_dot`.
2. Verify all four inner product axioms for this specific pair of vectors
   and an arbitrary third vector $\mathbf{x}$ and scalar $c$ of your choice.

In [ ]:
# Your solution here


### Exercise 2 (Medium)

Show that $f(t) = 1$ and $g(t) = t$ are **orthogonal** under the integral
inner product on $[-1, 1]$:

$$\langle f, g \rangle = \int_{-1}^{1} 1 \cdot t \, dt.$$

1. Compute the integral by hand (it is the integral of an odd function
   over a symmetric interval).
2. Verify numerically using `integral_ip`.
3. Are $f(t) = 1$ and $h(t) = t^2$ orthogonal on $[-1, 1]$? Why or why not?

In [ ]:
# Your solution here


### Exercise 3 (Challenge)

Let $W = \text{span}\{(1, 0, 1, 0),\; (0, 1, 0, 1)\}$ in $\mathbb{R}^4$
with the standard dot product.

1. Find a basis for $W^\perp$ using `orthogonal_complement_basis`.
2. Verify that $\dim(W) + \dim(W^\perp) = 4$.
3. Express $\mathbf{v} = (3, 2, 1, 4)$ as $\mathbf{v} = \mathbf{v}_W + \mathbf{v}_{W^\perp}$
   where $\mathbf{v}_W \in W$ and $\mathbf{v}_{W^\perp} \in W^\perp$.
   *Hint:* Project $\mathbf{v}$ onto the orthogonal basis of $W$.

In [ ]:
# Your solution here


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Inner product** | Any bilinear, symmetric, positive-definite function $\langle \cdot, \cdot \rangle$ |
| **Four axioms** | Symmetry, additivity, homogeneity, positive definiteness |
| **Weighted dot product** | $\langle \mathbf{u}, \mathbf{v} \rangle_w = \sum w_i u_i v_i$ with $w_i > 0$ |
| **Integral inner product** | $\langle f, g \rangle = \int_a^b f(t) g(t)\, dt$ for continuous functions |
| **Induced norm** | $\|\mathbf{v}\| = \sqrt{\langle \mathbf{v}, \mathbf{v} \rangle}$ |
| **Induced distance** | $d(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\|$ |
| **Orthogonality** | $\langle \mathbf{u}, \mathbf{v} \rangle = 0$; depends on the choice of inner product |
| **Orthogonal complement** | $W^\perp = \{\mathbf{v} : \langle \mathbf{v}, \mathbf{w} \rangle = 0 \; \forall \mathbf{w} \in W\}$; $\dim W + \dim W^\perp = \dim V$ |